In [ ]:
!pip install -q --upgrade kaggle
!pip install -q pandas numpy matplotlib scikit-learn sentence-transformers bertopic spacy
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import files
import os
import shutil

# Upload your Kaggle API file
uploaded = files.upload()

# Configure Kaggle credentials
os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully!")

Saving kaggle.json to kaggle.json
Kaggle API configured successfully!


In [ ]:
!pip -q install datasets

In [ ]:
from datasets import load_dataset

# Load dataset in streaming mode
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train",
    streaming=True
)

# Read only 5,000 records
records = []
for i, paper in enumerate(dataset):
    records.append(paper)
    if i + 1 >= 5000:
        break

print("Records loaded:", len(records))
print("Available fields:", records[0].keys() if records else "No records")

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

Records loaded: 5000
Available fields: dict_keys(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'])


In [ ]:
import pandas as pd

papers = pd.DataFrame(records)

print("Dataset shape:", papers.shape)
print("\nAvailable columns:")
print(papers.columns.tolist())

print("\nFirst 3 records:")
display(papers.head(3))

Dataset shape: (5000, 4)

Available columns:
['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract']

First 3 records:


,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...


In [ ]:
# Keep only the useful columns
papers = papers[["title", "abstract"]].copy()

# Remove missing values
papers = papers.dropna(subset=["title", "abstract"])

# Remove duplicate papers
papers = papers.drop_duplicates(subset=["title"])

# Remove empty titles or abstracts
papers = papers[
    (papers["title"].str.strip() != "") &
    (papers["abstract"].str.strip() != "")
]

# Reset row numbers
papers = papers.reset_index(drop=True)

print("Clean dataset shape:", papers.shape)
print("\nSample research paper:")
print("Title:", papers.loc[0, "title"])
print("\nAbstract:", papers.loc[0, "abstract"])

Clean dataset shape: (4978, 2)

Sample research paper:
Title: Learning from compressed observations

Abstract:   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate fun

In [ ]:
import re
import spacy

# Load English NLP model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs, punctuation, and numbers
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove stopwords and lemmatize
    doc = nlp(text)

    words = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_space and len(token) > 2
    ]

    return " ".join(words)

# Combine title and abstract for better topic modeling
papers["text"] = (
    papers["title"].astype(str) + " " +
    papers["abstract"].astype(str)
)

papers["clean_text"] = papers["text"].apply(preprocess_text)

print("Preprocessing completed!")
print("\nOriginal text:")
print(papers.loc[0, "text"][:300])

print("\nCleaned text:")
print(papers.loc[0, "clean_text"][:300])

Preprocessing completed!

Original text:
Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some s

Cleaned text:
learn compress observation problem statistical learning construct predictor random variable function relate random variable basis training sample joint distribution allowable predictor draw specify class goal approach asymptotically performance expect loss good predictor class consider setting perfe


In [ ]:
!pip -q install bertopic sentence-transformers

from bertopic import BERTopic

# Combine titles and abstracts
documents = (
    papers["title"].fillna("") + ". " +
    papers["abstract"].fillna("")
).tolist()

# Create and train BERTopic
topic_model = BERTopic(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    min_topic_size=20,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(documents)

# Save topic assignments
papers["topic"] = topics

print("BERTopic model trained successfully!")
print("Number of topics:", len(set(topics)) - (1 if -1 in topics else 0))

topic_model.get_topic_info().head(10)

2026-10-27 20:26:22,599 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

2026-10-27 20:35:40,774 - BERTopic - Embedding - Completed ✓
2026-10-27 20:35:40,777 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-27 20:36:14,900 - BERTopic - Dimensionality - Completed ✓
2026-10-27 20:36:14,902 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-27 20:36:15,133 - BERTopic - Cluster - Completed ✓
2026-10-27 20:36:15,141 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-27 20:36:15,852 - BERTopic - Representation - Completed ✓


BERTopic model trained successfully!
Number of topics: 51


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1639,-1_the_of_and_to,"[the, of, and, to, in, is, we, for, that, on]",[Scalable Similarity Learning using Large Marg...
1,0,247,0_reinforcement_policy_learning_the,"[reinforcement, policy, learning, the, to, in,...",[Model-Based Policy Gradients with Parameter-B...
2,1,208,1_clustering_means_clusters_cluster,"[clustering, means, clusters, cluster, data, t...",[Clustering with Transitive Distance and K-Mea...
3,2,174,2_bayesian_structure_networks_network,"[bayesian, structure, networks, network, varia...",[Parameterized Complexity Results for Exact Ba...
4,3,173,3_deep_neural_networks_of,"[deep, neural, networks, of, the, to, network,...",[On the Number of Linear Regions of Deep Neura...
5,4,128,4_bandit_regret_armed_bandits,"[bandit, regret, armed, bandits, arms, reward,...",[Parametrized Stochastic Multi-armed Bandits w...
6,5,125,5_classifiers_boosting_ensemble_class,"[classifiers, boosting, ensemble, class, class...",[RandomBoost: Simplified Multi-class Boosting ...
7,6,115,6_gradient_stochastic_descent_convergence,"[gradient, stochastic, descent, convergence, o...",[A Smoothing Stochastic Gradient Method for Co...
8,7,105,7_network_graph_networks_link,"[network, graph, networks, link, nodes, in, th...",[Active Learning for Node Classification in As...
9,8,95,8_online_regret_the_algorithm,"[online, regret, the, algorithm, we, bounds, a...",[A Generalized Online Mirror Descent with Appl...


In [ ]:
# Get topic information
topic_info = topic_model.get_topic_info()

print("Total topics found:", len(topic_info) - 1)  # Excludes outlier topic
display(topic_info.head(15))


Total topics found: 51


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1639,-1_the_of_and_to,"[the, of, and, to, in, is, we, for, that, on]",[Scalable Similarity Learning using Large Marg...
1,0,247,0_reinforcement_policy_learning_the,"[reinforcement, policy, learning, the, to, in,...",[Model-Based Policy Gradients with Parameter-B...
2,1,208,1_clustering_means_clusters_cluster,"[clustering, means, clusters, cluster, data, t...",[Clustering with Transitive Distance and K-Mea...
3,2,174,2_bayesian_structure_networks_network,"[bayesian, structure, networks, network, varia...",[Parameterized Complexity Results for Exact Ba...
4,3,173,3_deep_neural_networks_of,"[deep, neural, networks, of, the, to, network,...",[On the Number of Linear Regions of Deep Neura...
5,4,128,4_bandit_regret_armed_bandits,"[bandit, regret, armed, bandits, arms, reward,...",[Parametrized Stochastic Multi-armed Bandits w...
6,5,125,5_classifiers_boosting_ensemble_class,"[classifiers, boosting, ensemble, class, class...",[RandomBoost: Simplified Multi-class Boosting ...
7,6,115,6_gradient_stochastic_descent_convergence,"[gradient, stochastic, descent, convergence, o...",[A Smoothing Stochastic Gradient Method for Co...
8,7,105,7_network_graph_networks_link,"[network, graph, networks, link, nodes, in, th...",[Active Learning for Node Classification in As...
9,8,95,8_online_regret_the_algorithm,"[online, regret, the, algorithm, we, bounds, a...",[A Generalized Online Mirror Descent with Appl...


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Generate embeddings for our existing papers
if "embeddings" not in globals():
    embeddings = embedding_model.encode(
        (papers["title"] + ". " + papers["abstract"]).tolist(),
        show_progress_bar=True,
        batch_size=64,
        convert_to_numpy=True
    )

def search_papers(query, top_k=5):
    # Convert search query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Calculate similarity with all papers
    scores = cosine_similarity(query_embedding, embeddings)[0]

    # Get the most similar papers
    best_indices = np.argsort(scores)[::-1][:top_k]

    results = papers.iloc[best_indices][["title", "abstract"]].copy()
    results["similarity_score"] = scores[best_indices].round(3)

    return results.reset_index(drop=True)

print("Semantic similarity search is ready!")

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

Semantic similarity search is ready!


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Save the paper dataset
papers[["title", "abstract", "topic"]].to_csv(
    "papers.csv", index=False
)

# Save embeddings used by semantic search
np.save("embeddings.npy", embeddings)

# Download both files
files.download("papers.csv")
files.download("embeddings.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>